import library

In [74]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
nltk.download('punkt')
nltk.download('stopwords')
from langdetect import detect 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rahel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rahel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


baca file dataset

In [75]:
df = pd.read_csv('snatia-dataset.csv')
df_before = df.copy() # untuk perbandingan data awal dan akhir

deteksi bahasa

In [76]:
def language(text):
    try: 
        return detect(text)
    except:
        return "error"

filter hanya bahasa indonesia

In [77]:
df['bahasa'] = df ['comment'].apply(language) # kolom bahasa 
df = df[df['bahasa'] == 'id'].copy()

preprocessing

membersihkan komentar

In [78]:
def clean_text(text):
    text = text.lower()  # Ubah ke huruf kecil
    text = text.translate(str.maketrans('', '', string.punctuation)) # Hapus tanda baca
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Hapus URL
    text = re.sub(r'\d+', '', text)  # Hapus angka
    text = re.sub(r'\s+', ' ', text).strip()  # Hapus spasi berlebih
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)  # Hapus mention
    text = re.sub(r'#', '', text)  # Hapus hashtag
    text = re.sub(r'[^\w\s]', '', text) # hapus simbol

    return text

df['comment_clean'] = df['comment'].apply(clean_text)

baca kamus normalisasi dan membuat dictionary normalisasi

In [79]:
df_kamusalay = pd.read_csv('kamus_alay.csv', names =['slang', 'formal'])
normalisasi = dict(zip(df_kamusalay['slang'], df_kamusalay['formal']))

# fungsi normalisasi untuk mengganti kata slang dengan kata formal
def normalize(text):
    tokens = text.split()
    normalized = [normalisasi.get(word, word) for word in tokens]
    return ' '.join(normalized)

df['comment_clean'] = df['comment_clean'].apply(normalize)

In [80]:
df['tokenized'] = df['comment_clean'].apply(word_tokenize)

In [81]:
df['unique_tokens'] = df['tokenized'].apply(lambda x: list(set(x)))  # Mengubah token menjadi tipe data set untuk menghilangkan duplikasi

stopwords nltk dan satrawi

In [82]:
indo_stopwords = set(stopwords.words('indonesian'))
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()
all_stopwords = list(indo_stopwords.union(set(sastrawi_stopwords)))

Penghapusan tanda baca dan lowercase

In [83]:
def remove_stopwords(tokens, stopwords_list):
    return [word for word in tokens if word not in stopwords_list]

df['remove_stopwords'] = df['unique_tokens'].apply(lambda x: remove_stopwords(x, all_stopwords))

stemming

In [84]:
stemfactory = StemmerFactory()
stemmer = stemfactory.create_stemmer()

df['stemmed'] = df['remove_stopwords'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])

In [85]:
df['terms'] = df['stemmed'].apply(lambda x: ' '.join(x))

simpan file processing

In [86]:
df.to_csv('processed_data.csv', index=False)
print(f"Jumlah data awal: {len(df_before)} baris")
print(f"Jumlah data setelah pembersihan: {len(df)} baris")

Jumlah data awal: 1000 baris
Jumlah data setelah pembersihan: 839 baris
